# `py_trees`로 로봇 파츠/여러 로봇의 동작 순서 조율하기 (탐색용)

**아직 `src/n2o`에 통합되지 않은 탐색/학습 노트북입니다.** [`py_trees`](https://github.com/splintered-reality/py_trees)
(behavior tree 라이브러리)의 기본 사용법을, n2o의 실제 `Part` 구현체(`SO101Arm`/`AmazingHand`)를 대상으로 익힙니다.
이 노트북에서 `Robot.router()`/`N2O.run()`은 전혀 바뀌지 않습니다 -- `py_trees`를 그 자리에 끼워 넣을지는 이후에
결정합니다.

## 왜 스케줄러/플래너가 필요한가

뇌파 신호 자체가 시간에 따라 순차적으로 들어오는 것은 이미 `N2O.run()`의 cycle 루프(`signal.read()` → `decoder()`
→ 정착 대기, 한 번에 하나씩)가 처리하고 있는 문제라서, 그 자체는 새 스케줄러가 필요한 이유가 아닙니다. 실제 빈
자리는 **디코딩된 명령 하나를 여러 파츠(또는 여러 로봇)에 걸쳐 어떤 순서/조건으로 실행할지**입니다 -- 지금
`Robot.router()`(`src/n2o/robot/__init__.py`)는 명령에 담긴 모든 파츠를 한 번에 스레드로 병렬 디스패치만 할 뿐,
"팔이 먼저 특정 위치에 도달한 다음에만 손을 쥐게 한다"거나 "실패하면 대체 제스처를 시도한다"거나 "스테이션 A가
끝난 다음 스테이션 B를 움직인다" 같은 순서/분기를 표현할 방법이 없습니다.

`Part.done_event`(`src/n2o/robot/part.py`)는 이미 이런 "future cross-part coordinator"를 염두에 두고 만들어진
훅입니다 -- 지금은 `Robot.router()`만 세팅/클리어하지만, docstring이 예고한 대로 다른 코드가 `done_event.wait()`/
`.is_set()`으로 파츠 하나의 완료를 기다릴 수 있게 되어 있습니다. 이 노트북 4절에서 `py_trees`의 커스텀
`Behaviour`가 바로 그 훅을 폴링하는 예시를 만들어 봅니다.

## 설치

`py_trees`는 이 노트북을 위해 `examples` 의존성 그룹에 새로 추가되었습니다 (`pyproject.toml`, `uv add --group
examples py_trees`) -- `mujoco`/`lerobot`처럼 노트북 전용 패키지라 core `dependencies`엔 들어가지 않습니다.

```bash
uv sync --group examples
uv run --group examples jupyter lab examples/10_py_trees_task_coordination.ipynb
```

In [1]:
import threading
import time

import py_trees

from n2o.robot import Robot
from n2o.robot.arm.so101 import SO101Arm
from n2o.robot.hand.amazing_hand_right import AmazingHand

# 실물 하드웨어에 연결하지 않습니다 -- port를 비워두면 `_real_arm`/`_servo`가 `None`으로 남고,
# 이 노트북에서 쓰는 `goal()`/`done_event`는 둘 다 I/O를 하지 않으므로 안전합니다.
arm = SO101Arm()
hand = AmazingHand()

## 1. `py_trees` 기본 개념 -- `Behaviour`와 `Status`

`py_trees`의 모든 노드는 `py_trees.behaviour.Behaviour`를 상속하고 `update()`를 오버라이드합니다. `update()`는
`py_trees.common.Status.SUCCESS`/`FAILURE`/`RUNNING` 중 하나를 반환해야 합니다 -- 트리를 한 번 "틱(tick)"할 때마다
루트부터 이 상태들이 전파됩니다. 가장 단순한 예시부터 봅니다.

In [2]:
class Hello(py_trees.behaviour.Behaviour):
    def update(self):
        print(f"  [{self.name}] tick")
        return py_trees.common.Status.SUCCESS


hello = Hello(name="hello")
hello.tick_once()
print("status:", hello.status)

  [hello] tick
status: Status.SUCCESS


## 2. `Sequence`로 순서 지정 -- 실제 `SO101Arm`/`AmazingHand`의 `goal()` 사용

`Sequence`는 자식을 왼쪽부터 순서대로 틱합니다 -- 하나가 `FAILURE`면 그 자리에서 멈추고, 모두 `SUCCESS`여야
`Sequence` 자신도 `SUCCESS`가 됩니다. 아래 `MoveTo`는 `Part.goal(cmd)`(순수 계산, I/O 없음 -- `move()`가 아닙니다)를
호출해 그 제스처의 목표 값을 조회하고, 알 수 없는 제스처면 `FAILURE`를 돌려줍니다.

In [3]:
class MoveTo(py_trees.behaviour.Behaviour):
    """`part.goal(cmd)`를 조회하는 behaviour -- I/O 없이 목표값만 계산하므로 실물 하드웨어 없이도
    안전하게 틱할 수 있습니다. 실제로 움직이려면 `goal` 대신 `move`를 부르면 되지만(§4 참고), 그건
    시간이 걸리는 동작이라 `RUNNING`을 다뤄야 합니다."""

    def __init__(self, name, part, cmd):
        super().__init__(name)
        self.part = part
        self.cmd = cmd

    def update(self):
        try:
            target = self.part.goal(self.cmd)
        except ValueError as exc:
            print(f"  [{self.name}] FAILURE ({exc})")
            return py_trees.common.Status.FAILURE
        print(f"  [{self.name}] SUCCESS -> target={target}")
        return py_trees.common.Status.SUCCESS


routine = py_trees.composites.Sequence(
    name="팔 올리고 손 쥐기",
    memory=True,
    children=[
        MoveTo("arm: up", arm, "up"),
        MoveTo("hand: grip", hand, "grip"),
    ],
)
py_trees.trees.BehaviourTree(routine).tick()
print("\nrouting status:", routine.status)
print(py_trees.display.unicode_tree(routine))

  [arm: up] SUCCESS -> target={'shoulder_pan': 87.6, 'shoulder_lift': -23.25, 'elbow_flex': 128.79, 'wrist_flex': -85.58, 'wrist_roll': 38.11}
  [hand: grip] SUCCESS -> target=[1.4, 0.0, 1.4, 0.0, 1.4, 0.0, 1.4, 0.0]

routing status: Status.SUCCESS
{-} 팔 올리고 손 쥐기
    --> arm: up
    --> hand: grip



## 3. `Selector`로 대체 동작(fallback) 지정

`Selector`는 반대로, 자식을 왼쪽부터 틱하다 하나가 `SUCCESS`면 그 자리에서 멈춥니다 -- "우선 이걸 시도하고, 실패하면
저걸 시도한다"는 폴백 로직입니다. 여기서는 존재하지 않는 손 제스처를 먼저 시도해 일부러 `FAILURE`를 내고,
`Selector`가 두 번째 자식(`"release"`, 실제 `GESTURES`에 있는 제스처)으로 넘어가는 것을 봅니다.

In [4]:
fallback = py_trees.composites.Selector(
    name="손 제스처 시도 (없으면 release로 대체)",
    memory=False,
    children=[
        MoveTo("try: 정의되지 않은 제스처", hand, "clap"),
        MoveTo("fallback: release", hand, "release"),
    ],
)
fallback.tick_once()
print("\nfallback status:", fallback.status)
print(py_trees.display.unicode_tree(fallback))

  [try: 정의되지 않은 제스처] FAILURE (unknown AmazingHand gesture: 'clap')
  [fallback: release] SUCCESS -> target=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

fallback status: Status.SUCCESS
[o] 손 제스처 시도 (없으면 release로 대체)
    --> try: 정의되지 않은 제스처
    --> fallback: release



## 4. `Parallel` + `Part.done_event`로 여러 파츠를 동시에, 비동기로 조율

지금까지는 `goal()`만 썼기 때문에 매 틱이 즉시 끝났습니다. 실물 `move()`처럼 시간이 걸리는 동작을 표현하려면
`update()`가 아직 끝나지 않았을 때 `RUNNING`을 반환해야 합니다 -- `py_trees`는 다음 틱에 같은 자리부터 다시
`update()`를 부릅니다.

`Robot.router()`는 이미 각 파츠를 스레드로 동시에 돌리고, `Part.done_event`를 그 파츠가 끝나면 set합니다
(`src/n2o/robot/__init__.py`의 `_dispatch()`). 아래 `MoveAsync`는 그 `done_event`를 그대로 재사용합니다 -- 백그라운드
스레드에서 `duration_s`초 후 `done_event.set()`을 흉내내고, `update()`는 그게 set될 때까지 `RUNNING`을 반환합니다.
`Parallel`(정책 `SuccessOnAll`)로 감싸면 팔과 손이 동시에 움직이되, 트리 쪽에서 "둘 다 끝났는지"를 명시적으로
기다릴 수 있습니다.

In [5]:
class MoveAsync(py_trees.behaviour.Behaviour):
    """`part.done_event`를 폴링하는 behaviour. `initialise()`(이 behaviour가 새로 RUNNING에 진입할 때
    한 번만 호출됨)에서 `done_event`를 클리어하고, `duration_s`초 후 set하는 백그라운드 스레드를
    띄웁니다 -- 실물 `move()`가 걸리는 시간을 흉내낸 것일 뿐, 실제 시리얼 I/O는 없습니다."""

    def __init__(self, name, part, duration_s):
        super().__init__(name)
        self.part = part
        self.duration_s = duration_s

    def initialise(self):
        self.part.done_event.clear()
        threading.Thread(
            target=lambda: (time.sleep(self.duration_s), self.part.done_event.set()),
            daemon=True,
        ).start()

    def update(self):
        if self.part.done_event.is_set():
            print(f"  [{self.name}] SUCCESS (동작 완료)")
            return py_trees.common.Status.SUCCESS
        print(f"  [{self.name}] RUNNING (아직 이동 중)")
        return py_trees.common.Status.RUNNING


parallel = py_trees.composites.Parallel(
    name="팔 + 손 동시에 움직이기",
    policy=py_trees.common.ParallelPolicy.SuccessOnAll(),
    children=[
        MoveAsync("arm: 이동 중 (0.6s)", arm, duration_s=0.6),
        MoveAsync("hand: 이동 중 (0.3s)", hand, duration_s=0.3),
    ],
)
tree = py_trees.trees.BehaviourTree(parallel)
while parallel.status != py_trees.common.Status.SUCCESS:
    tree.tick()
    time.sleep(0.1)
print("\nparallel status:", parallel.status)

  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)
  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)


  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)
  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] SUCCESS (동작 완료)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [arm: 이동 중 (0.6s)] SUCCESS (동작 완료)



parallel status: Status.SUCCESS


## 5. 여러 대 로봇의 순서 지정

`Robot`은 arm/hand/camera를 묶는 평범한 컨테이너라서, 물리적으로 분리된 스테이션이 여러 대면 `Robot()` 인스턴스를
여러 개 만들면 됩니다. 지금 `Robot.router()`는 인스턴스 하나 안의 파츠만 다루고, 인스턴스 여러 개 사이의 순서는
아무 데도 정의돼 있지 않습니다 -- 여기서는 스테이션 A와 스테이션 B를 각각 `Robot()`으로 만들고, `Sequence` 안에
`Parallel` 두 개(스테이션 하나당 하나, §4의 `MoveAsync` 재사용)를 넣어서 "A가 완전히 끝난 다음에만 B가 시작"하는
순서를 표현합니다.

In [6]:
station_a = Robot()
station_a.arm = SO101Arm()
station_a.hand = AmazingHand()

station_b = Robot()
station_b.arm = SO101Arm()
station_b.hand = AmazingHand()


def station_routine(name, station, arm_cmd, hand_cmd):
    return py_trees.composites.Parallel(
        name=name,
        policy=py_trees.common.ParallelPolicy.SuccessOnAll(),
        children=[
            MoveAsync(f"{name}: arm {arm_cmd}", station.arm, duration_s=0.4),
            MoveAsync(f"{name}: hand {hand_cmd}", station.hand, duration_s=0.2),
        ],
    )


multi_robot_plan = py_trees.composites.Sequence(
    name="스테이션 A 먼저, 그 다음 스테이션 B",
    memory=True,
    children=[
        station_routine("station A", station_a, "up", "grip"),
        station_routine("station B", station_b, "down", "release"),
    ],
)

tree = py_trees.trees.BehaviourTree(multi_robot_plan)
while multi_robot_plan.status != py_trees.common.Status.SUCCESS:
    tree.tick()
    time.sleep(0.1)
print("\n" + py_trees.display.unicode_tree(multi_robot_plan))

  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] RUNNING (아직 이동 중)
  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] RUNNING (아직 이동 중)


  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] SUCCESS (동작 완료)
  [station A: arm up] RUNNING (아직 이동 중)
  [station A: arm up] SUCCESS (동작 완료)
  [station B: arm down] RUNNING (아직 이동 중)


  [station B: hand release] RUNNING (아직 이동 중)
  [station B: arm down] RUNNING (아직 이동 중)
  [station B: hand release] RUNNING (아직 이동 중)
  [station B: arm down] RUNNING (아직 이동 중)
  [station B: hand release] SUCCESS (동작 완료)


  [station B: arm down] RUNNING (아직 이동 중)
  [station B: arm down] SUCCESS (동작 완료)

{-} 스테이션 A 먼저, 그 다음 스테이션 B
    /_/ station A
        --> station A: arm up
        --> station A: hand grip
    /_/ station B
        --> station B: arm down
        --> station B: hand release



## 6. 대기(wait) vs 선진행(proceed) -- 파츠별 상시 큐 (B 방식)

2절~5절은 전부 같은 패턴이었습니다 -- **command 하나 = tree를 새로 만들고, 끝날 때까지 기다렸다가 다음
command**(A 방식). 이 패턴에서는 구조적으로 항상 전체가 동기화됩니다 -- 다음 command를 만들 수 있는지 자체가
"이번 tree가 끝났는가"에 묶여 있어서, 파츠 하나가 먼저 끝나도 다음 command 자체가 아직 존재하지 않기 때문입니다.

파츠마다 페이스를 다르게(느린 파츠는 계속 진행 중이어도 빠른 파츠는 벌써 다음 command를 시작) 가려면, tree를
command마다 새로 만드는 대신 **파츠당 behaviour를 하나만 만들어서 계속 tick하고, 그 behaviour가 자기 몫의
command들을 큐에 쌓아뒀다 순서대로 소비**해야 합니다(B 방식).

여기서 자연스러운 질문이 나옵니다 -- 파츠별 큐가 서로 독립적이면, "같은 command에 속한 다른 파츠가 아직 안
끝났는지"는 어떻게 판단하나? **같은 behaviour(tree)를 다시 쓰는 게 아니라, 같은 command에서 나온 파츠별 큐
아이템들이 작은 공유 객체 하나(`CommandGroup`)를 함께 참조하는 것으로 풉니다.** 파츠별 큐는 독립적으로 계속
tick되지만, 아이템 하나를 끝낼 때마다 자신이 속한 `CommandGroup`에 "나 끝났다"고 표시하고, 다음 아이템으로
넘어가기 전에 그 그룹의 `wait` 정책이 켜져 있으면 "다른 파츠도 다 끝났는지"를 그 공유 객체에게 물어봅니다.
`wait=False`면 아예 안 물어보고 바로 다음 아이템으로 넘어갑니다 -- 이게 지난 대화에서 나온 "느린 장치가 끝날
때까지 기다리게 하거나, 빠른 장치는 다음 명령을 받거나"를 파츠별 옵션(`wait`)으로 표현한 것입니다.

In [7]:
import collections


class CommandGroup:
    """같은 command에서 나온 파츠별 큐 아이템들이 함께 참조하는 공유 배리어. `PartQueue`는
    서로 독립적으로 tick되지만, "같은 command에 속한 다른 파츠가 아직 안 끝났는지"는 파츠
    하나만 봐서는 알 수 없으므로, 그 command가 만들어질 때 딱 한 번 만들어서 관련된 모든
    파츠의 큐 아이템에 함께 실어 보냅니다."""

    def __init__(self, parts, wait):
        self.wait = wait
        self.pending = set(parts)  # 아직 이 command를 못 끝낸 파츠 이름들

    def mark_done(self, part_name):
        self.pending.discard(part_name)

    def blocks_next(self, part_name):
        """`part_name`이 자기 큐의 다음 아이템으로 넘어가도 되는지."""
        if not self.wait:
            return False
        return bool(self.pending - {part_name})


class PartQueue(py_trees.behaviour.Behaviour):
    """파츠 하나를 계속 담당하는 상시 behaviour -- §2~5와 달리 command마다 새로 만들어지지
    않고, 이 셀에서 한 번만 만들어져 계속 tick됩니다. 자기 큐에 쌓인 (cmd, group,
    duration_s)를 순서대로 소비하되, 방금 끝낸 아이템의 `CommandGroup.blocks_next()`가
    True면(= wait 정책이고 같은 command의 다른 파츠가 아직 안 끝남) 다음 아이템을 시작하지
    않고 기다립니다. `part.done_event`로 "언제 끝났는지"를 아는 건 §4의 `MoveAsync`와 같은
    방식입니다."""

    def __init__(self, name, part_name, part):
        super().__init__(name)
        self.part_name = part_name
        self.part = part
        self.queue = collections.deque()
        self._current = None
        self._last_group = None

    def enqueue(self, cmd, group, duration_s):
        self.queue.append((cmd, group, duration_s))

    def update(self):
        if self._current is None:
            if self._last_group is not None and self._last_group.blocks_next(
                self.part_name
            ):
                print(f"  [{self.name}] WAITING (같은 command의 다른 파츠 대기 중)")
                return py_trees.common.Status.RUNNING
            if not self.queue:
                return py_trees.common.Status.RUNNING  # 할 일 없음 -- 다음 command 대기
            cmd, group, duration_s = self.queue.popleft()
            print(f"  [{self.name}] START {cmd!r}")
            self._current = (cmd, group)
            self.part.done_event.clear()
            threading.Thread(
                target=lambda: (time.sleep(duration_s), self.part.done_event.set()),
                daemon=True,
            ).start()
            return py_trees.common.Status.RUNNING

        cmd, group = self._current
        if self.part.done_event.is_set():
            print(f"  [{self.name}] DONE {cmd!r}")
            group.mark_done(self.part_name)
            self._last_group = group
            self._current = None
        return py_trees.common.Status.RUNNING

아래에서 같은 시나리오를 `wait=True`/`wait=False` 두 정책으로 각각 돌립니다: **command 1**이 팔(`"up"`, 0.6초 -- 느림)과
손(`"grip"`, 0.2초 -- 빠름)을 같은 `CommandGroup`으로 묶어서 동시에 지시하고, 그 직후 **command 2**가 손만
(`"release"`, 0.2초) 지시합니다. 손은 command 1의 자기 몫을 팔보다 훨씬 먼저 끝내므로, `wait` 정책에 따라 command
2를 언제 시작하는지가 갈립니다.

In [8]:
def run_demo(wait_for_group):
    demo_arm = SO101Arm()
    demo_hand = AmazingHand()
    arm_queue = PartQueue("arm queue", "arm", demo_arm)
    hand_queue = PartQueue("hand queue", "hand", demo_hand)

    # command 1: 팔(느림, 0.6s) + 손(빠름, 0.2s)을 같은 command로 -- wait 정책의 대상
    cmd1 = CommandGroup(parts={"arm", "hand"}, wait=wait_for_group)
    arm_queue.enqueue("up", cmd1, duration_s=0.6)
    hand_queue.enqueue("grip", cmd1, duration_s=0.2)

    # command 2: 손만 대상 (팔은 애초에 이 command와 무관)
    cmd2 = CommandGroup(parts={"hand"}, wait=wait_for_group)
    hand_queue.enqueue("release", cmd2, duration_s=0.2)

    robot_parts = py_trees.composites.Parallel(
        name="로봇 파츠 (상시 tick)",
        policy=py_trees.common.ParallelPolicy.SuccessOnAll(),
        children=[arm_queue, hand_queue],
    )
    tree = py_trees.trees.BehaviourTree(robot_parts)

    def busy():
        return bool(
            arm_queue.queue
            or arm_queue._current
            or hand_queue.queue
            or hand_queue._current
        )

    while busy():
        tree.tick()
        time.sleep(0.05)


print(
    "=== wait=True: 손이 command 1을 먼저 끝내도, 팔이 끝날 때까지 command 2(release)를 미룸 ==="
)
run_demo(wait_for_group=True)

print(
    "\n=== wait=False: 손은 자기 큐를 독립적으로 소비 -- 팔이 아직 안 끝났어도 바로 command 2를 시작 ==="
)
run_demo(wait_for_group=False)

=== wait=True: 손이 command 1을 먼저 끝내도, 팔이 끝날 때까지 command 2(release)를 미룸 ===
  [arm queue] START 'up'
  [hand queue] START 'grip'


  [hand queue] DONE 'grip'
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)


  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)
  [hand queue] WAITING (같은 command의 다른 파츠 대기 중)


  [arm queue] DONE 'up'
  [hand queue] START 'release'


  [hand queue] DONE 'release'

=== wait=False: 손은 자기 큐를 독립적으로 소비 -- 팔이 아직 안 끝났어도 바로 command 2를 시작 ===
  [arm queue] START 'up'
  [hand queue] START 'grip'


  [hand queue] DONE 'grip'
  [hand queue] START 'release'


  [hand queue] DONE 'release'
  [arm queue] DONE 'up'


## 정리

5절까지(A 방식)로 `Sequence`/`Selector`/`Parallel`만으로도 파츠 간 순서, 실패 시 대체 동작, 여러 `Robot` 인스턴스
간 순서를 표현할 수 있었습니다. 6절(B 방식)이 보여준 건, 그중 "대기 vs 선진행"만큼은 A로 표현이 안 된다는
점입니다 -- command 발행이 tree 하나의 완료에 묶여 있는 한, 빠른 파츠가 먼저 끝나도 다음 command 자체가 아직
존재하지 않으니까요. B는 파츠별 behaviour를 command마다 새로 만들지 않고 영속시키는 대신, "같은 command에 속한
파츠들이 서로의 진행 상태를 알아야 한다"는 문제가 새로 생기는데, 이건 behaviour 자체를 공유하는 게 아니라
`CommandGroup`처럼 작은 공유 객체를 큐 아이템에 함께 실어 보내는 걸로 풉니다.

**이 노트북이 하지 않은 것:** `Command.translate()`의 출력을 실제로 `PartQueue.enqueue()` 호출로 바꾸는 부분,
`N2O.run()`의 cycle 루프가 (지금처럼 매 사이클 `router()`를 blocking 호출하는 대신) 매 사이클 `enqueue()`만 하고
바로 다음 디코딩으로 넘어가는 부분, 그리고 `wait`을 파츠별/커맨드별로 어떻게 설정하게 할지(생성자 인자?
`CommandConfig`? 등)는 아직 정해지지 않았습니다 -- `ROADMAP.md`가 예고한 "future cross-part coordinator" 자리에
이 설계가 맞는지 확인하고 나서 진행할 문제입니다.